# Bronze Layer Validation
**Purpose**: Validate raw data ingestion completeness and quality
- Row count verification
- Schema inspection
- Partition validation
- Consolidate the files into one delta table

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import * 
from datetime import datetime

StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 20, Finished, Available, Finished, False)

In [19]:
spark = SparkSession.builder.appName("BronzeLayerValidation").getOrCreate()

StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 21, Finished, Available, Finished, False)

In [13]:
from pyspark.sql.functions import col, to_timestamp, month, year
from pyspark.sql.types import *

# 1️⃣ Define explicit schema 
from pyspark.sql.types import *

bronze_schema = StructType([
    StructField("Site_ID", StringType(), True),
    StructField("Province_Code", StringType(), True),
    StructField("Province", StringType(), True),
    StructField("Vendor", StringType(), True),
    StructField("Technology", StringType(), True),
    StructField("Outage_Start", TimestampType(), True),
    StructField("Outage_End", TimestampType(), True),
    StructField("Duration_Minutes", DoubleType(), True),
    StructField("Availability_Percent", DoubleType(), True),
    StructField("Cause", StringType(), True),
    StructField("month", IntegerType(), True),
    StructField("year", IntegerType(), True)
])


# 2️⃣ Read with schema (NO inferSchema)
bronze_df = (
    spark.read
         .option("header", "true")
         .schema(bronze_schema)
         .csv("Files/network_events/")
)


StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 15, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql.functions import current_timestamp

bronze_df = bronze_df.withColumn("_load_timestamp", current_timestamp())


In [25]:
#Quick validation(check if lineage is intact and if all months data are loaded)

#Validate all months are loaded
bronze_df.select("year", "month").distinct().orderBy("year", "month").show()

#Validate row counts per month
bronze_df.groupBy("year", "month").count().orderBy("year", "month").show()



StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 27, Finished, Available, Finished, False)

+----+-----+
|year|month|
+----+-----+
|2024|    1|
|2024|    2|
|2024|    3|
|2024|    4|
|2024|    5|
|2024|    6|
+----+-----+

+----+-----+-----+
|year|month|count|
+----+-----+-----+
|2024|    1| 8000|
|2024|    2| 8000|
|2024|    3| 8000|
|2024|    4| 8000|
|2024|    5| 7001|
|2024|    6| 5662|
+----+-----+-----+



In [26]:
bronze_df.printSchema()
bronze_df.show(5)

StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 28, Finished, Available, Finished, False)

root
 |-- Site_ID: string (nullable = true)
 |-- Province_Code: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Vendor: string (nullable = true)
 |-- Technology: string (nullable = true)
 |-- Outage_Start: timestamp (nullable = true)
 |-- Outage_End: timestamp (nullable = true)
 |-- Duration_Minutes: double (nullable = true)
 |-- Availability_Percent: double (nullable = true)
 |-- Cause: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+------------+-------------+-------------+------+----------+-------------------+-------------------+----------------+--------------------+--------+----+-----+
|     Site_ID|Province_Code|     Province|Vendor|Technology|       Outage_Start|         Outage_End|Duration_Minutes|Availability_Percent|   Cause|year|month|
+------------+-------------+-------------+------+----------+-------------------+-------------------+----------------+--------------------+--------+----+-----+
|SA-SITE-

In [27]:
#Write to table
(
    bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_network_events")
)


StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 29, Finished, Available, Finished, False)

In [28]:
#Validate data in table
spark.table("bronze_network_events").count()
spark.table("bronze_network_events").printSchema()



StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 30, Finished, Available, Finished, False)

root
 |-- Site_ID: string (nullable = true)
 |-- Province_Code: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Vendor: string (nullable = true)
 |-- Technology: string (nullable = true)
 |-- Outage_Start: timestamp (nullable = true)
 |-- Outage_End: timestamp (nullable = true)
 |-- Duration_Minutes: double (nullable = true)
 |-- Availability_Percent: double (nullable = true)
 |-- Cause: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)



In [29]:
spark.table("bronze_network_events").count()

StatementMeta(, 7c5461c9-24b0-4c65-ad03-58b6c59cb1cb, 31, Finished, Available, Finished, False)

44663